In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/content/drive/MyDrive/Nairobi_Air_Quality_cleaned.csv")
df.head()

,location,timestamp,P0,P1,P2,humidity,temperature
0,76,2026-01-01 00:00:00+00:00,11.600000,22.400000,20.450000,93.338095,18.438095
1,76,2026-01-01 01:00:00+00:00,9.850000,19.500000,17.600000,93.555000,18.470000
2,76,2026-01-01 02:00:00+00:00,10.952381,21.952381,19.523810,94.304762,18.180952
3,76,2026-01-01 03:00:00+00:00,11.823529,23.176471,20.647059,95.070588,17.788235
4,76,2026-01-01 04:00:00+00:00,9.333333,20.333333,17.333333,92.666667,17.900000


In [3]:
#add lags
cols_to_add_lags = ["P0","P1","P2","humidity","temperature"]

for col in cols_to_add_lags:
  df[f"{col}_lag_1h"] = df.groupby("location")[col].shift(1)
  df[f"{col}_lag_2h"] = df.groupby("location")[col].shift(2)
  df[f"{col}_lag_3h"] = df.groupby("location")[col].shift(3)

  df = df.dropna().reset_index(drop=True)

In [4]:
df.head()

,location,timestamp,P0,P1,P2,humidity,temperature,P0_lag_1h,P0_lag_2h,P0_lag_3h,...,P1_lag_3h,P2_lag_1h,P2_lag_2h,P2_lag_3h,humidity_lag_1h,humidity_lag_2h,humidity_lag_3h,temperature_lag_1h,temperature_lag_2h,temperature_lag_3h
0,76,2026-01-01 17:00:00+00:00,11.50,20.50,18.30,72.190,20.150,7.40,5.25,5.333333,...,8.00,12.75,8.1875,7.666667,66.515,60.625,57.333333,21.185,22.60625,22.833333
1,76,2026-01-01 18:00:00+00:00,13.25,23.15,20.95,74.710,19.440,11.50,7.40,5.250000,...,9.25,18.30,12.7500,8.187500,72.190,66.515,60.625000,20.150,21.18500,22.606250
2,76,2026-01-01 19:00:00+00:00,16.80,29.10,26.15,77.785,18.615,13.25,11.50,7.400000,...,14.65,20.95,18.3000,12.750000,74.710,72.190,66.515000,19.440,20.15000,21.185000
3,76,2026-01-01 20:00:00+00:00,14.80,26.10,24.15,80.340,18.335,16.80,13.25,11.500000,...,20.50,26.15,20.9500,18.300000,77.785,74.710,72.190000,18.615,19.44000,20.150000
4,76,2026-01-01 21:00:00+00:00,11.35,20.30,18.75,81.345,18.270,14.80,16.80,13.250000,...,23.15,24.15,26.1500,20.950000,80.340,77.785,74.710000,18.335,18.61500,19.440000


In [5]:
from statsmodels.tsa.stattools import adfuller

core_variables = ["P0", "P1", "P2", "humidity", "temperature"]


for loc_id in df["location"].unique():
    print(f"\n=== Stationarity for Location {loc_id} ===")

    for var in core_variables:

        series_data = df[df["location"] == loc_id][var].dropna()

        # Run the test on the clean slice
        result = adfuller(series_data)

        print(
            f"  Variable: {var:<12} | ADF Stat: {result[0]:.4f} | p-value: {result[1]:.6f}"
        )


=== Stationarity for Location 76 ===
  Variable: P0           | ADF Stat: -12.2421 | p-value: 0.000000
  Variable: P1           | ADF Stat: -18.8908 | p-value: 0.000000
  Variable: P2           | ADF Stat: -19.7982 | p-value: 0.000000
  Variable: humidity     | ADF Stat: -2.6810 | p-value: 0.077383
  Variable: temperature  | ADF Stat: -6.3222 | p-value: 0.000000

=== Stationarity for Location 3966 ===
  Variable: P0           | ADF Stat: -3.3295 | p-value: 0.013611
  Variable: P1           | ADF Stat: -9.1056 | p-value: 0.000000
  Variable: P2           | ADF Stat: -9.1437 | p-value: 0.000000
  Variable: humidity     | ADF Stat: -4.1643 | p-value: 0.000757
  Variable: temperature  | ADF Stat: -9.7353 | p-value: 0.000000

=== Stationarity for Location 3967 ===
  Variable: P0           | ADF Stat: -9.3096 | p-value: 0.000000
  Variable: P1           | ADF Stat: -34.0933 | p-value: 0.000000
  Variable: P2           | ADF Stat: -36.6663 | p-value: 0.000000
  Variable: humidity     | ADF S

In [6]:
weather_cols = [
    "humidity",
    "temperature",
    "humidity_lag_1h",
    "humidity_lag_2h",
    "humidity_lag_3h",
    "temperature_lag_1h",
    "temperature_lag_2h",
    "temperature_lag_3h",
]

for loc_id in df["location"].unique():
    mask = df["location"] == loc_id

    for var in weather_cols:
        df.loc[mask, var] = df.loc[mask, var].diff()

df = df.dropna().reset_index(drop=True)

In [7]:
df.head()

,location,timestamp,P0,P1,P2,humidity,temperature,P0_lag_1h,P0_lag_2h,P0_lag_3h,...,P1_lag_3h,P2_lag_1h,P2_lag_2h,P2_lag_3h,humidity_lag_1h,humidity_lag_2h,humidity_lag_3h,temperature_lag_1h,temperature_lag_2h,temperature_lag_3h
0,76,2026-01-01 18:00:00+00:00,13.25,23.15,20.95,2.520,-0.710,11.50,7.40,5.25,...,9.25,18.30,12.75,8.1875,5.675,5.890,3.291667,-1.035,-1.42125,-0.227083
1,76,2026-01-01 19:00:00+00:00,16.80,29.10,26.15,3.075,-0.825,13.25,11.50,7.40,...,14.65,20.95,18.30,12.7500,2.520,5.675,5.890000,-0.710,-1.03500,-1.421250
2,76,2026-01-01 20:00:00+00:00,14.80,26.10,24.15,2.555,-0.280,16.80,13.25,11.50,...,20.50,26.15,20.95,18.3000,3.075,2.520,5.675000,-0.825,-0.71000,-1.035000
3,76,2026-01-01 21:00:00+00:00,11.35,20.30,18.75,1.005,-0.065,14.80,16.80,13.25,...,23.15,24.15,26.15,20.9500,2.555,3.075,2.520000,-0.280,-0.82500,-0.710000
4,76,2026-01-01 22:00:00+00:00,9.15,17.25,15.30,1.015,-0.265,11.35,14.80,16.80,...,29.10,18.75,24.15,26.1500,1.005,2.555,3.075000,-0.065,-0.28000,-0.825000


In [8]:
# Confirm if the values are stationary
core_variables = ["humidity", "temperature"]


for loc_id in df["location"].unique():
    print(f"\n=== Stationarity for Location {loc_id} ===")

    for var in core_variables:

        series_data = df[df["location"] == loc_id][var].dropna()

        # Run the test on the clean slice
        result = adfuller(series_data)

        print(
            f"  Variable: {var:<12} | ADF Stat: {result[0]:.4f} | p-value: {result[1]:.6f}"
        )


=== Stationarity for Location 76 ===
  Variable: humidity     | ADF Stat: -17.8106 | p-value: 0.000000
  Variable: temperature  | ADF Stat: -16.8205 | p-value: 0.000000

=== Stationarity for Location 3966 ===
  Variable: humidity     | ADF Stat: -11.0419 | p-value: 0.000000
  Variable: temperature  | ADF Stat: -9.6740 | p-value: 0.000000

=== Stationarity for Location 3967 ===
  Variable: humidity     | ADF Stat: -11.3807 | p-value: 0.000000
  Variable: temperature  | ADF Stat: -12.8134 | p-value: 0.000000

=== Stationarity for Location 3981 ===
  Variable: humidity     | ADF Stat: -19.0925 | p-value: 0.000000
  Variable: temperature  | ADF Stat: -18.8415 | p-value: 0.000000


In [9]:
df.to_csv("/content/drive/MyDrive/Nairobi_feature_engineered_dataset.csv", index=False)